# 01 — Collect data

Two source families (see `config.SOURCES`):
- **`opensubtitles_enfi`** (role: `pairs`) — real EN→FI parallel dialogue from OPUS.
  Broad translation ability; Finnish leans colloquial but is translator-normalized.
- **`genius_rap`** (role: `flavor`) — Finnish rap lyrics via the Genius API. Pure
  Helsinki puhekieli/slang, but **FI-only** (no English side).

The rap lyrics can't train a translator by themselves (no pairs), so here we just
collect + clean them. In `01b_synthesize.ipynb` we manufacture the missing English
side by back-translation. Everything runs locally.

> Personal/public-web use. Respect Genius API terms; don't redistribute lyrics.

In [ ]:
from puhekieli_llm.config import summary, SOURCES, RAP_ARTISTS
from puhekieli_llm import sources as S
print(summary())
print()
for name, m in SOURCES.items():
    print(f"{name:20s} role={m['role']:7s} register={m['register']:9s} {m['status']}")

## A) Genius rap lyrics (flavor, FI-only)

Get a free token at <https://genius.com/api-clients> and set it in the environment
**before** launching Jupyter (so it never lands in the notebook):

```bash
export GENIUS_ACCESS_TOKEN="..."
```

Then fetch. This hits the network and is rate-limited — run once; raw json is
cached in `data/raw/genius_rap/` so you can re-clean without re-fetching.

In [ ]:
import os
if os.getenv('GENIUS_ACCESS_TOKEN'):
    S.fetch_genius_lyrics(RAP_ARTISTS, max_songs_per_artist=40)
else:
    print('No GENIUS_ACCESS_TOKEN set — skipping fetch.')
    print('Set it and re-run, or clean already-cached raw json below.')

In [ ]:
from puhekieli_llm.config import RAW, CLEAN
# Clean whatever raw lyrics we have -> data/clean/genius_rap.jsonl
if (RAW / 'genius_rap').exists() and any((RAW / 'genius_rap').glob('*.json')):
    S.clean_genius_lyrics()
else:
    print('No raw lyrics yet. Fetch first (needs a Genius token).')

In [ ]:
# Peek + puhekieli sanity check on what we collected
from puhekieli_llm.eval import puhekieli_score, corpus_puhekieli_rate
p = CLEAN / 'genius_rap.jsonl'
if p.exists():
    recs = list(S.read_jsonl(p))
    print(f'{len(recs)} lyric lines')
    for r in recs[:8]:
        print(f"  [{puhekieli_score(r['fi'])['score']:+.1f}] {r['fi']}")
    print(f"\nspoken-leaning fraction: {corpus_puhekieli_rate([r['fi'] for r in recs]):.0%}")
else:
    print('genius_rap.jsonl not built yet.')

## B) OpenSubtitles EN-FI (parallel pairs)

Streamed straight from OPUS (moses zip, ~870 MB, downloaded once into
`data/raw/` which is gitignored). We read the two aligned members directly from
the zip and cap the count for a laptop-friendly demo. First run downloads the
zip; be patient. Writes `data/clean/opensubtitles_enfi.jsonl`.

In [ ]:
# Set small while iterating; bump up when you're ready to train for real.
# First call downloads ~870 MB into data/raw/ (cached afterward). Set
# FETCH_OPENSUBS=1 in the env to actually download; otherwise this cell skips
# (keeps headless verification fast and avoids surprise downloads).
import os
from puhekieli_llm.config import RAW
MAX_PAIRS = 50_000
_zip = RAW / 'opensubtitles_enfi' / 'en-fi.txt.zip'
if os.getenv('FETCH_OPENSUBS') or _zip.exists():
    S.fetch_opensubtitles(max_pairs=MAX_PAIRS)
else:
    print('Skipping OpenSubtitles download (~870 MB).')
    print('Set FETCH_OPENSUBS=1 and re-run this cell to fetch.')

In [ ]:
q = CLEAN / 'opensubtitles_enfi.jsonl'
if q.exists():
    recs = list(S.read_jsonl(q))
    print(f'{len(recs)} EN-FI pairs')
    for r in recs[:6]:
        print(f"  EN: {r['en']}\n  FI: {r['fi']}\n")

## ✅ Next
- `01b_synthesize.ipynb` — back-translate rap lyrics into EN→FI pairs (local LLM).
- then `02_tokenizer.ipynb` — train a BPE tokenizer over subtitles + rap + synth.